In [3]:
pip install numpy pandas scikit-learn matplotlib seaborn shap urllib3

In [5]:
import os
import zipfile
import urllib.request
import ssl  # Added to handle the SSL certificate error
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, f1_score

import shap

# Create directory structure for downloads/outputs
OUTPUT_DIR = "bank_marketing_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"[*] Directories initialized. Outputs will be saved to: {OUTPUT_DIR}/")

# -------------------------------------------------------------------------
# STEP 1: DOWNLOAD AND EXTRACT DATASET (WITH SSL BYPASS)
# -------------------------------------------------------------------------
URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
ZIP_PATH = os.path.join(OUTPUT_DIR, "bank_marketing.zip")
EXTRACT_PATH = os.path.join(OUTPUT_DIR, "extracted")

if not os.path.exists(ZIP_PATH):
    print("[*] Downloading Bank Marketing dataset from UCI Repository...")
    try:
        # Create an unverified context to bypass the expired SSL certificate issue
        ssl_context = ssl._create_unverified_context()
        with urllib.request.urlopen(URL, context=ssl_context) as response, open(ZIP_PATH, 'wb') as out_file:
            out_file.write(response.read())
        print("[+] Download complete.")
    except Exception as e:
        print(f"[-] Download failed: {e}")
        print("[!] Please try downloading the file manually from UCI and placing it in the output folder.")
        raise e

if not os.path.exists(EXTRACT_PATH):
    print("[*] Extracting dataset files...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)

    # Nested zip extraction for bank-additional
    nested_zip = os.path.join(EXTRACT_PATH, "bank-additional.zip")
    if os.path.exists(nested_zip):
        with zipfile.ZipFile(nested_zip, 'r') as zip_ref:
            zip_ref.extractall(EXTRACT_PATH)
    print("[+] Extraction complete.")

# Load the bank-additional-full dataset
data_file = os.path.join(EXTRACT_PATH, "bank-additional", "bank-additional-full.csv")
if not os.path.exists(data_file):
    data_file = os.path.join(EXTRACT_PATH, "bank", "bank-full.csv")

print(f"[*] Loading data from: {data_file}")
df = pd.read_csv(data_file, sep=';')

# -------------------------------------------------------------------------
# STEP 2: EXPLORATORY DATA ANALYSIS & VISUALIZATION DOWNLOADING
# -------------------------------------------------------------------------
print("[*] Performing EDA and saving exploratory visualizations...")

plt.figure(figsize=(6, 4))
sns.countplot(x='y', data=df, palette='Set2')
plt.title('Target Variable Distribution')
plt.xlabel('Subscribed (y)')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '01_target_distribution.png'))
plt.close()

plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='age', hue='y', multiple='stack', bins=30, palette='viridis')
plt.title('Age Distribution by Subscription Status')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '02_age_distribution.png'))
plt.close()

# -------------------------------------------------------------------------
# STEP 3: CATEGORICAL ENCODING & PREPROCESSING
# -------------------------------------------------------------------------
print("[*] Encoding features and splitting data...")

df['y'] = df['y'].map({'yes': 1, 'no': 0})

categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

X = df.drop(columns=['y'])
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

# -------------------------------------------------------------------------
# STEP 4: MODEL TRAINING AND EVALUATION
# -------------------------------------------------------------------------
print("[*] Training models...")

models = {
    "Logistic_Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random_Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

metrics_summary = []
plt.figure(figsize=(10, 8))

for name, model in models.items():
    print(f"    -> Training {name}...")
    if name == "Logistic_Regression":
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

    f1 = f1_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'{name} Confusion Matrix\nF1-Score: {f1:.4f}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'03_{name}_confusion_matrix.png'))
    plt.close()

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    plt.figure(1)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.4f})')

    metrics_summary.append(f"=== {name} ===\n" + classification_report(y_test, y_pred) + f"\nAUC Score: {roc_auc:.4f}\n\n")

plt.figure(1)
plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '04_combined_roc_curve.png'))
plt.close()

with open(os.path.join(OUTPUT_DIR, "performance_metrics.txt"), "w") as f:
    f.writelines(metrics_summary)

print("[+] Models evaluated and logs exported.")

# -------------------------------------------------------------------------
# STEP 5: SHAP MODEL EXPLANATIONS (5 PREDICTIONS)
# -------------------------------------------------------------------------
print("[*] Generating SHAP explanations for 5 test predictions...")

explainer = shap.TreeExplainer(models["Random_Forest"])
sample_indices = [10, 25, 45, 80, 100]
X_sample = X_test.iloc[sample_indices]
shap_values = explainer.shap_values(X_sample)

for i, idx in enumerate(sample_indices):
    plt.figure(figsize=(10, 4))
    shap.plots.bar(shap.Explanation(values=shap_values[..., 1][i],
                                    base_values=explainer.expected_value[1],
                                    data=X_sample.iloc[i],
                                    feature_names=X.columns), max_display=10, show=False)
    plt.title(f"SHAP Profile #{i+1} (Row Index: {idx})")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'05_shap_prediction_sample_{i+1}.png'))
    plt.close()

plt.figure(figsize=(10, 6))
shap_values_total = explainer.shap_values(X_test.iloc[:200])
shap.summary_plot(shap_values_total[..., 1], X_test.iloc[:200], show=False)
plt.title("Global SHAP Feature Importance Summary", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '06_shap_global_summary.png'))
plt.close()

print(f"\n[Done] All sorted! Run complete. Check the folder '{OUTPUT_DIR}' for all assets.")

[*] Directories initialized. Outputs will be saved to: bank_marketing_outputs/
[*] Downloading Bank Marketing dataset from UCI Repository...
[+] Download complete.
[*] Extracting dataset files...
[+] Extraction complete.
[*] Loading data from: bank_marketing_outputs/extracted/bank-additional/bank-additional-full.csv
[*] Performing EDA and saving exploratory visualizations...


/tmp/ipykernel_1900/1732082293.py:69: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(x='y', data=df, palette='Set2')


[*] Encoding features and splitting data...
[*] Training models...
    -> Training Logistic_Regression...
    -> Training Random_Forest...
[+] Models evaluated and logs exported.
[*] Generating SHAP explanations for 5 test predictions...

[Done] All sorted! Run complete. Check the folder 'bank_marketing_outputs' for all assets.
